In [ ]:
"""
MULTI-STOCK OPTIMIZER - TRAIN SEPARATE MODELS FOR EACH STOCK
=============================================================
This script trains and saves separate optimized models for each stock

Usage:
    python multi_stock_optimizer.py

Output:
    - optimized_models_U.pkl
    - optimized_models_AMD.pkl
    - optimized_models_IONQ.pkl
    - optimization_results_U.json
    - optimization_results_AMD.json
    - optimization_results_IONQ.json
"""

import sys
import os
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# Setup paths
#PROJECT_PATH = os.path.dirname(os.path.abspath(__file__))
#sys.path.insert(0, PROJECT_PATH)

from optimize_with_model_persistence import PersistentModelOptimizer


class MultiStockOptimizer:
    """
    Train and optimize separate models for multiple stocks
    """
    
    def __init__(self, stock_configs):
        """
        Initialize multi-stock optimizer
        
        Args:
            stock_configs: dict of {ticker: {'data_file': path, 'target_metric': metric}}
        """
        self.stock_configs = stock_configs
        self.optimization_results = {}
        
    def optimize_single_stock(self, ticker, config, 
                              prediction_horizons=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
                                 entry_thresholds=[0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.3, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
                                 take_profit_pcts=[3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0, 9.0, 10.0],
                                 stop_loss_pcts=[-2.0, -2.5, -3.0, -3.5, -4.0, -4.5, -5.0, -5.5, -6.0, -7.0],
                                 max_holding_days_list=[5,6,7,8,9, 10,11,12,13, 14, 20],
                                 profit_windows=[3, 4, 5, 6, 7],
                                 n_iterations=200,):
        """
        Optimize models for a single stock
        """
        
        print("\n" + "="*80)
        print(f"🎯 OPTIMIZING MODELS FOR {ticker}")
        print("="*80)
        print(f"Data file: {config['data_file']}")
        print(f"Target metric: {config['target_metric']}")
        print(f"Iterations: {n_iterations}")
        print("="*80)
        
        try:
            # Create optimizer for this stock
            optimizer = PersistentModelOptimizer(
                data_file=config['data_file'],
                target_metric=config['target_metric']
            )
            
            # File paths with ticker name
            model_save_path = f"optimized_models_{ticker}.pkl"
            results_save_path = f"optimization_results_{ticker}.json"
            
            # Run optimization
            best_result = optimizer.optimize_and_save_models(
                prediction_horizons=prediction_horizons,
                entry_thresholds=entry_thresholds,
                take_profit_pcts=take_profit_pcts,
                stop_loss_pcts=stop_loss_pcts,
                max_holding_days_list=max_holding_days_list,
                profit_windows=profit_windows,
                n_iterations=n_iterations,
                model_save_path=model_save_path,
                results_save_path=results_save_path
            )
            
            # Store results
            self.optimization_results[ticker] = {
                'success': True,
                'model_path': model_save_path,
                'results_path': results_save_path,
                'best_params': best_result['params'],
                'performance': {
                    'total_return': float(best_result['total_return']),
                    'sharpe_ratio': float(best_result['sharpe_ratio']),
                    'max_drawdown': float(best_result['max_drawdown']),
                    'win_rate': float(best_result['win_rate']),
                    'num_trades': int(best_result['num_trades'])
                }
            }
            
            print(f"\n✅ {ticker} OPTIMIZATION COMPLETE!")
            print(f"   Model saved: {model_save_path}")
            print(f"   Results saved: {results_save_path}")
            print(f"   Total Return: {best_result['total_return']:.1f}%")
            print(f"   Sharpe Ratio: {best_result['sharpe_ratio']:.2f}")
            print(f"   Win Rate: {best_result['win_rate']:.1f}%")
            
            return True
            
        except Exception as e:
            print(f"\n❌ {ticker} OPTIMIZATION FAILED!")
            print(f"   Error: {str(e)}")
            
            self.optimization_results[ticker] = {
                'success': False,
                'error': str(e)
            }
            
            return False
    
    def optimize_all_stocks(self, **optimization_params):
        """
        Optimize models for all stocks in the configuration
        """
        
        print("\n" + "="*80)
        print("🚀 MULTI-STOCK MODEL OPTIMIZATION")
        print("="*80)
        print(f"Stocks to optimize: {list(self.stock_configs.keys())}")
        print(f"Total stocks: {len(self.stock_configs)}")
        print("="*80)
        
        successful = 0
        failed = 0
        
        for ticker, config in self.stock_configs.items():
            success = self.optimize_single_stock(ticker, config, **optimization_params)
            
            if success:
                successful += 1
            else:
                failed += 1
        
        # Summary
        print("\n" + "="*80)
        print("📊 MULTI-STOCK OPTIMIZATION SUMMARY")
        print("="*80)
        print(f"✅ Successful: {successful}/{len(self.stock_configs)}")
        print(f"❌ Failed: {failed}/{len(self.stock_configs)}")
        
        if successful > 0:
            print("\n🏆 SUCCESSFUL OPTIMIZATIONS:")
            for ticker, result in self.optimization_results.items():
                if result['success']:
                    perf = result['performance']
                    print(f"\n{ticker}:")
                    print(f"   Model: {result['model_path']}")
                    print(f"   Return: {perf['total_return']:.1f}%")
                    print(f"   Sharpe: {perf['sharpe_ratio']:.2f}")
                    print(f"   Win Rate: {perf['win_rate']:.1f}%")
                    print(f"   Trades: {perf['num_trades']}")
        
        if failed > 0:
            print("\n❌ FAILED OPTIMIZATIONS:")
            for ticker, result in self.optimization_results.items():
                if not result['success']:
                    print(f"   {ticker}: {result['error']}")
        
        # Save summary
        self._save_multi_stock_summary()
        
        print("\n" + "="*80)
        print("✅ MULTI-STOCK OPTIMIZATION COMPLETE!")
        print("="*80)
        
        return self.optimization_results
    
    def _save_multi_stock_summary(self):
        """Save summary of all optimizations"""
        
        summary = {
            'optimization_date': datetime.now().isoformat(),
            'total_stocks': len(self.stock_configs),
            'successful': sum(1 for r in self.optimization_results.values() if r['success']),
            'failed': sum(1 for r in self.optimization_results.values() if not r['success']),
            'results': self.optimization_results
        }
        
        filename = f"multi_stock_optimization_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        with open(filename, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\n💾 Multi-stock summary saved: {filename}")


# =============================================================================
# CONFIGURATION
# =============================================================================

def create_stock_configs():
    """
    Define your stock configurations here
    
    Returns:
        dict of {ticker: config}
    """
    
    configs = {
        #'U': {
        #    'data_file': '111025/U_daily.csv',
        #   'target_metric': 'log_price_ratio_sma20'  # Can customize per stock
        #},
        #'AMD': {
        #    'data_file': '111025/AMD_daily.csv',
        #    'target_metric': 'log_price_ratio_sma20'
        #},
        'AVGO': {
            'data_file': '111025/AVGO_daily.csv',
            'target_metric': 'log_price_ratio_sma20'
        },
        #'IONQ': {
        #    'data_file': '111025/IONQ_daily.csv',
        #    'target_metric': 'log_price_ratio_sma20'
        #}
        
    }
    
    return configs


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    """
    Main execution - optimize models for all stocks
    """
    
    print("\n" + "="*80)
    print("🎯 MULTI-STOCK MODEL OPTIMIZER")
    print("="*80)
    print("This will train separate optimized models for each stock")
    print("="*80)
    
    # Get stock configurations
    stock_configs = create_stock_configs()
    
    print("\n📋 STOCKS TO OPTIMIZE:")
    for ticker, config in stock_configs.items():
        print(f"   {ticker:6s} - {config['data_file']}")
    
    # Confirm
    print("\n⚠️  This will take time (30-60 minutes per stock)")
    print("   Each stock gets its own optimized model")
    response = input("\nProceed? (yes/no): ")
    
    if response.lower() not in ['yes', 'y']:
        print("❌ Optimization cancelled")
        exit(0)
    
    # Create optimizer
    optimizer = MultiStockOptimizer(stock_configs)
    
    # Run optimization for all stocks
    # You can adjust these parameters based on your needs
    results = optimizer.optimize_all_stocks(
        prediction_horizons=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
                                 entry_thresholds=[0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.3, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
                                 take_profit_pcts=[3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0, 9.0, 10.0],
                                 stop_loss_pcts=[-2.0, -2.5, -3.0, -3.5, -4.0, -4.5, -5.0, -5.5, -6.0, -7.0],
                                 max_holding_days_list=[5,6,7,8,9, 10,11,12,13, 14, 20],
                                 profit_windows=[3, 4, 5, 6, 7],
                                 n_iterations=200,  # Increase to 200-300 for better results
    )
    
    # Display final summary
    print("\n" + "="*80)
    print("🎉 ALL OPTIMIZATIONS COMPLETE!")
    print("="*80)
    print("\n📁 FILES CREATED:")
    
    for ticker, result in results.items():
        if result['success']:
            print(f"\n{ticker}:")
            print(f"   📦 {result['model_path']}")
            print(f"   📊 {result['results_path']}")
    
    print("\n" + "="*80)
    print("✅ Ready for daily trading!")
    print("   Use: python multi_stock_signal_generator.py")
    print("="*80)



🎯 MULTI-STOCK MODEL OPTIMIZER
This will train separate optimized models for each stock

📋 STOCKS TO OPTIMIZE:
   AVGO   - 111025/AVGO_daily.csv

⚠️  This will take time (30-60 minutes per stock)
   Each stock gets its own optimized model
